In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import wandb

wandb.login(key="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")

wandb.init(
    project="24f3004935-t22026"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f3004935 (24f3004935-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run siifiktq
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260704_204946-siifiktq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run legendary-field-63
wandb: ⭐️ View project at https://wandb.ai/24f3004935-dl-genai-project/24f3004935-t22026
wandb: 🚀 View run at https://wandb.ai/24f3004935-dl-genai-project/24f3004935-t22026/runs/siifiktq


In [4]:
Train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(Train.shape)
print(test.shape)

Train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
Train["text"] = (
    "Question: " + Train["prompt"].fillna("") +
    " A: " + Train["A"].fillna("") +
    " B: " + Train["B"].fillna("") +
    " C: " + Train["C"].fillna("") +
    " D: " + Train["D"].fillna("") +
    " E: " + Train["E"].fillna("")
)

test["text"] = (
    "Question: " + test["prompt"].fillna("") +
    " A: " + test["A"].fillna("") +
    " B: " + test["B"].fillna("") +
    " C: " + test["C"].fillna("") +
    " D: " + test["D"].fillna("") +
    " E: " + test["E"].fillna("")
)

In [6]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {v:k for k,v in label2id.items()}

Train["label"] = Train["answer"].map(label2id)

In [7]:
Train.head(2)

,id,prompt,A,B,C,D,E,answer,text,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,Question: Pick the best possible answer: What ...,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,Question: What is accelerator-based light-ion ...,0


### Train-Test split

In [8]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(
    Train,
    test_size=0.2,
    random_state=42,
    stratify=Train["answer"]
)

## SCRATCH

In [9]:
import re
import math
import random
from collections import Counter

import wandb

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score

### Building Vocabulary

In [10]:
# Building the vocabulary from training data
counter = Counter()

for text in Train["text"]:
    words = re.findall(r"\b\w+\b", text.lower())
    counter.update(words)

# Special tokens
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

# Adding the words to vocabulary
for word, _ in counter.items():
    vocab[word] = len(vocab) # dict

# Reverse lookup 
id2word = {idx: word for word, idx in vocab.items()}

print("Vocabulary Size:", len(vocab))
print(list(vocab.items())[:15])

Vocabulary Size: 2983
[('<PAD>', 0), ('<UNK>', 1), ('question', 2), ('pick', 3), ('the', 4), ('best', 5), ('possible', 6), ('answer', 7), ('what', 8), ('is', 9), ('martin', 10), ('heidegger', 11), ('s', 12), ('view', 13), ('on', 14)]


In [11]:
MAX_LEN = 256

def encode_text(text):
    # Tokenization
    words = re.findall(r"\b\w+\b", text.lower())

    # Converting words to ids
    ids = [
        vocab.get(word, vocab["<UNK>"])
        for word in words
    ]

    # Truncate
    ids = ids[:MAX_LEN]

    # Padding
    padding = [vocab["<PAD>"]] * (MAX_LEN - len(ids))
    ids = ids + padding

    return ids

In [12]:
sample = encode_text(Train["text"].iloc[0])

print(sample[:30])
print(len(sample))

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 4, 15, 16, 17, 18, 19, 20, 21, 4, 22, 23, 24, 10, 11, 25, 26, 27]
256


### Pytorch Dataset

In [13]:
class MCQDataset(Dataset):

    def __init__(self, dataframe, is_test=False):
        self.data = dataframe
        self.is_test = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        text = self.data.iloc[idx]["text"]

        input_ids = torch.tensor(
            encode_text(text),
            dtype=torch.long
        )

        # Attention mask
        attention_mask = (input_ids != vocab["<PAD>"]).long()

        if self.is_test:
            return {
                "input_ids": input_ids,
                "attention_mask": attention_mask
            }

        label = torch.tensor(
            self.data.iloc[idx]["label"],
            dtype=torch.long
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label
        }

In [14]:
train_dataset = MCQDataset(train)

val_dataset = MCQDataset(val)

test_dataset = MCQDataset(test,is_test=True)

sample = train_dataset[0]

print(sample.keys())
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["labels"])

dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([256])
torch.Size([256])
tensor(2)


### Dataloader

In [15]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    generator=torch.Generator().manual_seed(SEED)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [16]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 256])
torch.Size([16, 256])
torch.Size([16])


## Positional Encoding

In [17]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=256):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [18]:
pos = PositionalEncoding(d_model=128)

x = torch.zeros(2, 256, 128)

y = pos(x)

print(y.shape)

torch.Size([2, 256, 128])


### Transformer Classifier

In [19]:
class TransformerClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        hidden_dim=256,
        num_classes=5,
        dropout=0.2
    ):
        super().__init__()

        # Word Embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        # Positional Encoding
        self.pos_encoder = PositionalEncoding(embed_dim)

        # One Transformer Encoder Layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True
        )

        # Stack Multiple Encoder Layers
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            embed_dim,
            num_classes
        )


        # Forward 

    def forward(self, input_ids, attention_mask):
    
        # Word Embeddings
        x = self.embedding(input_ids)
    
        # Add Positional Encoding
        x = self.pos_encoder(x)
    
        # Transformer Encoder
        x = self.transformer(
            x,
            src_key_padding_mask=(attention_mask == 0)
        )
    
        # Masked Mean Pooling
        mask = attention_mask.unsqueeze(-1)
    
        x = x * mask
    
        x = x.sum(dim=1)
    
        lengths = mask.sum(dim=1).clamp(min=1)
    
        x = x / lengths
    
        # Dropout
        x = self.dropout(x)
    
        # Classification
        logits = self.classifier(x)
    
        return logits

In [20]:
model = TransformerClassifier(
    vocab_size=len(vocab)
)

batch = next(iter(train_loader))

logits = model(
    batch["input_ids"],
    batch["attention_mask"]
)

print(logits.shape)

torch.Size([16, 5])


### Training

In [21]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=0.01
)

In [22]:
EPOCHS = 23

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ---------------- Evaluation ----------------
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    model.train()

    # ---------------- WandB ----------------
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "val_accuracy": acc,
        "val_f1_score": f1,
        "learning_rate": optimizer.param_groups[0]["lr"]
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {acc:.4f} | "
        f"F1: {f1:.4f}"
    )

wandb.finish()

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/23 | Loss: 1.5738 | Accuracy: 0.3425 | F1: 0.2943
Epoch 2/23 | Loss: 1.4131 | Accuracy: 0.5275 | F1: 0.4512
Epoch 3/23 | Loss: 0.7813 | Accuracy: 0.9300 | F1: 0.9315
Epoch 4/23 | Loss: 0.1346 | Accuracy: 0.9975 | F1: 0.9976
Epoch 5/23 | Loss: 0.0274 | Accuracy: 1.0000 | F1: 1.0000
Epoch 6/23 | Loss: 0.0121 | Accuracy: 1.0000 | F1: 1.0000
Epoch 7/23 | Loss: 0.0077 | Accuracy: 1.0000 | F1: 1.0000
Epoch 8/23 | Loss: 0.0055 | Accuracy: 1.0000 | F1: 1.0000
Epoch 9/23 | Loss: 0.0043 | Accuracy: 1.0000 | F1: 1.0000
Epoch 10/23 | Loss: 0.0035 | Accuracy: 1.0000 | F1: 1.0000
Epoch 11/23 | Loss: 0.0028 | Accuracy: 1.0000 | F1: 1.0000
Epoch 12/23 | Loss: 0.0024 | Accuracy: 1.0000 | F1: 1.0000
Epoch 13/23 | Loss: 0.0021 | Accuracy: 1.0000 | F1: 1.0000
Epoch 14/23 | Loss: 0.0018 | Accuracy: 1.0000 | F1: 1.0000
Epoch 15/23 | Loss: 0.0016 | Accuracy: 1.0000 | F1: 1.0000
Epoch 16/23 | Loss: 0.0014 | Accuracy: 1.0000 | F1: 1.0000
Epoch 17/23 | Loss: 0.0012 | Accuracy: 1.0000 | F1: 1.0000
Epoch 

wandb: updating run metadata


Epoch 23/23 | Loss: 0.0007 | Accuracy: 1.0000 | F1: 1.0000


wandb: 
wandb: Run history:
wandb:         epoch ▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
wandb: learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:    train_loss █▇▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:  val_accuracy ▁▃▇████████████████████
wandb:  val_f1_score ▁▃▇████████████████████
wandb: 
wandb: Run summary:
wandb:         epoch 23
wandb: learning_rate 0.0002
wandb:    train_loss 0.00074
wandb:  val_accuracy 1
wandb:  val_f1_score 1
wandb: 
wandb: 🚀 View run legendary-field-63 at: https://wandb.ai/24f3004935-dl-genai-project/24f3004935-t22026/runs/siifiktq
wandb: ⭐️ View project at: https://wandb.ai/24f3004935-dl-genai-project/24f3004935-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260704_204946-siifiktq/logs


In [23]:
torch.save(model.state_dict(), "scratch_transformer.pth")

### Testing

In [24]:
model = TransformerClassifier(
    vocab_size=len(vocab),
    embed_dim=128,
    num_heads=4,
    hidden_dim=256,
    num_layers=2,
    num_classes=5
)

model.load_state_dict(torch.load("scratch_transformer.pth"))
model.to(device)

TransformerClassifier(
  (embedding): Embedding(2983, 128, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=128, out_features=5, bias=True)
)

In [25]:
model.eval()

predictions = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ) 

        probs = torch.softmax(logits, dim=1)

        top3 = torch.argsort(
            probs,
            dim=1,
            descending=True
        )[:, :3]

        predictions.extend(top3.cpu().numpy())

In [26]:
final_predictions = []

for pred in predictions:
    labels = [id2label[i] for i in pred]
    final_predictions.append(" ".join(labels))

## PRE-TRAINED MODEL

In [27]:
from transformers import AutoTokenizer

MODEL = "FacebookAI/roberta-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [28]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = logits.argmax(axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

In [29]:
from datasets import Dataset

# Hugging Face datasets
train_ds = Dataset.from_pandas(
    train[["text", "label"]]
)

val_ds = Dataset.from_pandas(
    val[["text", "label"]]
)

test_ds = Dataset.from_pandas(
    test[["text"]]
)

In [30]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize , batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [31]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=5
)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy="epoch",
    seed = 8,
    report_to="wandb"
)

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)
trainer.train()

trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")

print("Model successfully saved")

wandb: setting up run en8l4f9e
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260704_205100-en8l4f9e
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tough-serenity-23
wandb: ⭐️ View project at https://wandb.ai/24f3004935-dl-genai-project/huggingface
wandb: 🚀 View run at https://wandb.ai/24f3004935-dl-genai-project/huggingface/runs/en8l4f9e
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.471278,1.192921,0.850000,0.844121
2,0.077236,0.023031,0.995000,0.994208
3,0.001159,0.000494,1.000000,1.000000
4,0.000804,0.000372,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved


In [34]:
trainer.save_model("./pretrained_model")
tokenizer.save_pretrained("./pretrained_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./pretrained_model/tokenizer_config.json',
 './pretrained_model/tokenizer.json')

In [35]:
import numpy as np
import torch

pred = trainer.predict(test_ds)

probs = torch.softmax(
    torch.tensor(pred.predictions),
    dim=1
).numpy()

predictions = []

for p in probs:
    top3 = np.argsort(p)[::-1][:3]
    predictions.append(
        " ".join([id2label[i] for i in top3])
    )

# MILESTONE 1

In [36]:
# print(train['answer'].value_counts(ascending=False))
# freq_dist = list(train['answer'].value_counts(ascending=False))
# print(freq_dist[0]+freq_dist[-1])

In [37]:
# import string

# vocab = set()

# for text in train["prompt"]:
#     # lowercase
#     text = str(text).lower()

#     # remove standard punctuation
#     text = text.translate(
#         str.maketrans('', '', string.punctuation)
#     )

#     # split by whitespace
#     words = text.split()

#     vocab.update(words)

# print("Vocabulary Size:", len(vocab))

In [38]:
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
# # Row ID 1
# text = train.loc[train["id"] == 1, "prompt"].iloc[0]

# # lowercase + remove punctuation
# text = text.lower()
# text = text.translate(str.maketrans('', '', string.punctuation))

# # tokenize
# words = text.split()

# # remove stopwords
# filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

# print(filtered_words)
# print("Count:", len(filtered_words))

In [39]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# combined_text = (
#     train["prompt"].fillna("") + " " +
#     train["A"].fillna("") + " " +
#     train["B"].fillna("") + " " +
#     train["C"].fillna("") + " " +
#     train["D"].fillna("") + " " +
#     train["E"].fillna("")
# )

# vectorizer = TfidfVectorizer(stop_words="english")

# X = vectorizer.fit_transform(combined_text)

# print("Vocabulary Size:", len(vectorizer.get_feature_names_out()))

In [40]:
# from sklearn.metrics.pairwise import cosine_similarity

# row = train.iloc[0]   # Row ID 1 if first row

# prompt_vec = vectorizer.transform([row["prompt"]])
# A_vec = vectorizer.transform([row["A"]])

# score = cosine_similarity(prompt_vec, A_vec)[0][0]

# print(round(score, 4))

In [41]:
# correct = 0

# for _, row in train.iterrows():

#     prompt_vec = vectorizer.transform([row["prompt"]])

#     scores = {}

#     for option in ["A", "B", "C", "D", "E"]:

#         option_vec = vectorizer.transform([row[option]])

#         scores[option] = cosine_similarity(
#             prompt_vec,
#             option_vec
#         )[0][0]

#     predicted = max(scores, key=scores.get) # highest cosine similarity

#     if predicted == row["answer"]:
#         correct += 1

# accuracy = (correct / len(train)) * 100

# print(f"Accuracy: {accuracy:.2f}%")

In [42]:
# def map_at_3(actual, predicted):
#     try:
#         rank = predicted.index(actual) + 1
#         return 1.0 / rank
#     except ValueError:
#         return 0.0

# actual = "C"
# predicted = ["C", "A", "B"]

# score = map_at_3(actual, predicted)

# print(score)

In [43]:
# actual = "B"
# predicted = ["D", "B", "E"]

# score = map_at_3(actual, predicted)

# print(score)

In [44]:
# top3 = train["answer"].value_counts().index[:3].tolist()

# prediction = top3

# score = 0

# for ans in train["answer"]:

#     if ans == prediction[0]:
#         score += 1.0

#     elif ans == prediction[1]:
#         score += 0.5

#     elif ans == prediction[2]:
#         score += 1/3

# map3 = score / len(train)

# print("Top 3:", prediction)
# print("MAP@3:", round(map3, 4))

# MILESTONE 2

## Preprocessing

In [45]:
# from datasets import load_dataset

# # Load train.csv
# Train = load_dataset(
#     "csv",
#     data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# )["train"]

# # Create combined_text = prompt + " " + A
# Train = Train.map(
#     lambda x: {
#         "combined_text": x["prompt"] + " " + x["A"]
#     }
# )

# # Character length of row 51
# print(len(Train[51]["combined_text"]))

In [46]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# print(tokenizer.vocab_size)

In [47]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# print(tokenizer.sep_token)
# # print(tokenizer.sep_token_id)

In [48]:
# print(Train)
# print(type(Train))

In [49]:
# print(type(Train["prompt"]))
# print(type(Train["prompt"][0]))
# print(Train["prompt"][0])

In [50]:
# encodings = tokenizer(
#     list(Train["prompt"]),
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )

# print(encodings["input_ids"].shape)

In [51]:
# from transformers import AutoConfig

# config = AutoConfig.from_pretrained("bert-base-uncased")

# hidden_size = config.hidden_size
# num_heads = config.num_attention_heads

# head_dim = hidden_size // num_heads

# print("Hidden Size:", hidden_size)
# print("Attention Heads:", num_heads)
# print("Head Dimension:", head_dim)

In [52]:
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModel

# # Load tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# model = AutoModel.from_pretrained("bert-base-uncased")

# # Tokenize row 0 prompt (default settings)
# inputs = tokenizer(Train[0]["prompt"], return_tensors="pt")

# # Forward pass
# outputs = model(**inputs)

# # Print shape
# print(outputs.last_hidden_state.shape)

In [53]:
# # Extract CLS embedding (batch 0, token 0)
# cls_embedding = outputs.last_hidden_state[0, 0]

# # Sum first 5 values
# answer = cls_embedding[:5].sum().item()

# print(round(answer, 4))

In [54]:
# model = AutoModel.from_pretrained(
#     "bert-base-uncased",
#     output_attentions=True
# )

# inputs = tokenizer("Light-ion fusion is a technique.", return_tensors="pt")
# outputs = model(**inputs)

# tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
# fusion_index = tokens.index("fusion")

# attention = outputs.attentions[-1][0, 0]

# print(round(attention[0, fusion_index].item(), 4))

In [55]:
# from sentence_transformers import SentenceTransformer, util

# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# # Get prompt and Option B from row 0
# prompt = Train[0]["prompt"]
# option_b = Train[0]["B"]

# # Generate embeddings
# prompt_embedding = model.encode(prompt, convert_to_tensor=True)
# option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# # Cosine similarity
# similarity = util.cos_sim(prompt_embedding, option_b_embedding)

# print(round(similarity.item(), 4))

In [56]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# mini_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# options = ["A", "B", "C", "D", "E"]

# mini_map = 0
# improved = 0

# for row in Train:

#     prompt = row["prompt"]
#     answer = row["answer"]

#     # Pipeline 1 : TF-IDF

#     docs = [prompt] + [row[o] for o in options]

#     tfidf = TfidfVectorizer()
#     X = tfidf.fit_transform(docs)

#     sims = cosine_similarity(X[0], X[1:]).flatten()

#     tfidf_rank = [
#         options[i]
#         for i in sims.argsort()[::-1]
#     ]


#     # Pipeline 2 : MiniLM

#     prompt_emb = mini_model.encode(prompt, convert_to_tensor=True)
#     option_embs = mini_model.encode(
#         [row[o] for o in options],
#         convert_to_tensor=True
#     )

#     sims = util.cos_sim(prompt_emb, option_embs)[0].cpu().numpy()

#     mini_rank = [
#         options[i]
#         for i in sims.argsort()[::-1]
#     ]

#     # MAP@3
#     top3 = mini_rank[:3]

#     if answer in top3:
#         mini_map += 1 / (top3.index(answer) + 1)


#     # Improvement count
#     tfidf_hit = answer in tfidf_rank[:3]
#     mini_hit = answer in mini_rank[:3]

#     if (not tfidf_hit) and mini_hit:
#         improved += 1

# mini_map /= len(Train)

# print("MiniLM MAP@3:", round(mini_map, 4))
# print("Improved count:", improved)

In [57]:
# from transformers import pipeline
# # Initialize zero-shot classifier
# classifier = pipeline("zero-shot-classification")

# # Prompt from row index 1
# prompt = Train[1]["prompt"]

# # Candidate labels: Options A, B, C
# candidate_labels = [
#     Train[1]["A"],
#     Train[1]["B"],
#     Train[1]["C"]
# ]

# # Predict
# result = classifier(
#     prompt,
#     candidate_labels=candidate_labels
# )

# # Probability score of the top-ranked option
# print(round(result["scores"][0], 4))

In [58]:
# result_softmax = classifier(
#     prompt,
#     candidate_labels=candidate_labels
# )

# # This question (Independent Sigmoids)
# result_sigmoid = classifier(
#     prompt,
#     candidate_labels=candidate_labels,
#     multi_label=True
# )

# softmax_sum = sum(result_softmax["scores"])
# sigmoid_sum = sum(result_sigmoid["scores"])

# print("Softmax sum:", softmax_sum)
# print("Sigmoid sum:", sigmoid_sum)
# print("Absolute difference:", round(abs(softmax_sum - sigmoid_sum), 4))

In [59]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
# model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# text = (
#     f"Question: {Train[0]['prompt']}. "
#     f"Is the correct answer A: {Train[0]['A']} "
#     f"or B: {Train[0]['B']}? "
#     f"Answer with just the letter A or B."
# )

# inputs = tokenizer(text, return_tensors="pt")

# outputs = model.generate(
#     **inputs,
#     max_new_tokens=5
# )

# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# SUBMISSION

In [60]:
submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": final_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,id,Prediction
0,1,A C D
1,2,B E C
2,3,B D A
3,4,E C A
4,5,C B D


In [61]:
# submission = pd.DataFrame({
#     "id": test["id"],
#     "Prediction": predictions
# })

# submission.to_csv(
#     "submission.csv",
#     index=False
# )

# submission.head()